# 📖 Notebook 1: RSS Feed Crawling & Parsing

The first job of a news aggregator is **collecting articles** from the internet. Most news sites publish an RSS or Atom feed — a structured XML file listing their latest articles. Our crawler fetches these feeds on a schedule, parses them, and stores the articles in PostgreSQL.

## Learning Objectives

By the end of this notebook, you'll understand:
- What RSS feeds are and how they work
- How to parse RSS/Atom feeds with Python's `feedparser` library
- How to normalise messy feed data into clean database rows
- How to schedule crawls using Redis sorted sets
- How real aggregators handle failures and rate limits

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/news-aggregator
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `newsagg_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [1]:
import psycopg2
import psycopg2.extras
import redis
import feedparser
import hashlib
import json
import time
from datetime import datetime, timezone

# --- Connection settings (match docker-compose.yml) ---
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "newsagg_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db(); conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker-compose up -d")

try:
    r = get_redis(); r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")
    print("   Run: docker-compose up -d")

✅ Connected to PostgreSQL
✅ Connected to Redis


## 📡 What Is an RSS Feed?

**RSS** (Really Simple Syndication) is an XML format that websites use to publish a list of their latest articles. Instead of scraping HTML, we just ask the site for its feed and get structured data back.

```
Your Aggregator              News Website
     │                            │
     │── GET /feed/rss.xml ──────►│
     │                            │
     │◄── XML with articles ──────│
     │    (title, link, date,     │
     │     summary, author)       │
```

**Why RSS?**
- Structured: no need to parse messy HTML
- Polite: websites *want* you to read their feeds
- Universal: nearly every news site has one
- Lightweight: much smaller than a full webpage

Let's see what a real RSS feed looks like.

In [2]:
# Fetch and parse a real RSS feed using feedparser
# feedparser handles RSS 2.0, Atom, RSS 1.0 — all the formats

feed_url = "https://hnrss.org/frontpage"  # Hacker News front page

print(f"🌐 Fetching: {feed_url}")
print()

parsed = feedparser.parse(feed_url)

# Feed-level metadata
print(f"📰 Feed Title:   {parsed.feed.get('title', 'N/A')}")
print(f"🔗 Feed Link:    {parsed.feed.get('link', 'N/A')}")
print(f"📝 Description:  {parsed.feed.get('subtitle', 'N/A')[:80]}")
print(f"📊 Articles:     {len(parsed.entries)}")
print(f"⚠️  Errors:       {parsed.bozo_exception if parsed.bozo else 'None'}")
print()

# Show the first 3 articles
print("First 3 articles:")
print("=" * 70)
for i, entry in enumerate(parsed.entries[:3]):
    print(f"\n  [{i+1}] {entry.get('title', 'No title')}")
    print(f"      Link:      {entry.get('link', 'N/A')}")
    print(f"      Published: {entry.get('published', 'N/A')}")
    summary = entry.get('summary', 'N/A')[:100]
    print(f"      Summary:   {summary}...")

🌐 Fetching: https://hnrss.org/frontpage



📰 Feed Title:   Hacker News: Front Page
🔗 Feed Link:    https://news.ycombinator.com/
📝 Description:  Hacker News RSS
📊 Articles:     20
⚠️  Errors:       None

First 3 articles:

  [1] Got an Old Kindle? It Might Not Work Anymore
      Link:      https://www.nytimes.com/wirecutter/reviews/older-kindle-support-ending/
      Published: Sun, 19 Apr 2026 22:56:32 +0000
      Summary:   <p>Article URL: <a href="https://www.nytimes.com/wirecutter/reviews/older-kindle-support-ending/">ht...

  [2] 2,100 Swiss municipalities showing which provider handles their official email
      Link:      https://mxmap.ch/
      Published: Sun, 19 Apr 2026 22:55:47 +0000
      Summary:   <p>Related ongoing thread: <i>Swiss authorities want to reduce dependency on Microsoft</i> - <a href...

  [3] Ex-CEO, ex-CFO of bankrupt AI company charged with fraud
      Link:      https://www.reuters.com/legal/government/ex-ceo-ex-cfo-bankrupt-ai-company-charged-with-fraud-2026-04-17/
      Published: Sun, 19 Apr 202

## 🔧 Normalising Feed Data

Different feeds use different field names, date formats, and structures. Our crawler needs to **normalise** everything into a consistent format before storing it.

Common problems:
- Dates: `"Mon, 31 Mar 2026 10:00:00 GMT"` vs `"2026-03-31T10:00:00Z"` vs missing entirely
- Authors: `"John"` vs `{"name": "John"}` vs `None`
- Summaries: HTML tags mixed with text, or empty
- URLs: relative vs absolute paths

In [3]:
import re
from time import mktime

def normalise_entry(entry, feed_id: int) -> dict:
    """
    Convert a messy feedparser entry into a clean, consistent dict
    that matches our database schema.
    """
    # --- Title ---
    title = entry.get("title", "Untitled").strip()

    # --- URL ---
    url = entry.get("link", "").strip()

    # --- Summary: strip HTML tags ---
    raw_summary = entry.get("summary", "") or ""
    summary = re.sub(r"<[^>]+>", "", raw_summary).strip()
    summary = summary[:1000]  # truncate very long summaries

    # --- Author: feedparser sometimes returns a dict ---
    author = entry.get("author", "Unknown")
    if isinstance(author, dict):
        author = author.get("name", "Unknown")

    # --- Published date ---
    published_at = None
    if hasattr(entry, "published_parsed") and entry.published_parsed:
        published_at = datetime.fromtimestamp(
            mktime(entry.published_parsed), tz=timezone.utc
        )
    elif hasattr(entry, "updated_parsed") and entry.updated_parsed:
        published_at = datetime.fromtimestamp(
            mktime(entry.updated_parsed), tz=timezone.utc
        )

    # --- Content hash for deduplication (we'll use this in Notebook 2) ---
    text_to_hash = f"{title.lower()} {summary.lower()}"
    content_hash = hashlib.sha256(text_to_hash.encode()).hexdigest()

    # --- Word count ---
    word_count = len(summary.split())

    return {
        "feed_id": feed_id,
        "title": title,
        "url": url,
        "summary": summary,
        "author": author,
        "published_at": published_at,
        "content_hash": content_hash,
        "word_count": word_count,
    }

# Test normalisation on the entries we already fetched
print("Normalised articles:")
print("=" * 70)
for entry in parsed.entries[:3]:
    normalised = normalise_entry(entry, feed_id=8)  # 8 = Hacker News
    print(f"\n  Title:     {normalised['title'][:60]}")
    print(f"  Author:    {normalised['author']}")
    print(f"  Published: {normalised['published_at']}")
    print(f"  Hash:      {normalised['content_hash'][:16]}...")
    print(f"  Words:     {normalised['word_count']}")

Normalised articles:

  Title:     Got an Old Kindle? It Might Not Work Anymore
  Author:    eigenhombre
  Published: 2026-04-19 20:56:32+00:00
  Hash:      ab66fdff6ae35a26...
  Words:     11

  Title:     2,100 Swiss municipalities showing which provider handles th
  Author:    doener
  Published: 2026-04-19 20:55:47+00:00
  Hash:      f6a017d058d350bc...
  Words:     21

  Title:     Ex-CEO, ex-CFO of bankrupt AI company charged with fraud
  Author:    1vuio0pswjnm7
  Published: 2026-04-19 20:30:19+00:00
  Hash:      67465dd25660827d...
  Words:     11


## 💾 Storing Articles in PostgreSQL

Once we've normalised an article, we insert it into our `articles` table. We use `ON CONFLICT (url) DO NOTHING` so that re-crawling the same feed doesn't create duplicate rows.

Let's first look at the articles that were pre-loaded by `init.sql`, then add new ones from a live feed.

In [4]:
# See what's already in the database from init.sql
conn = get_db()
cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cursor.execute("""
    SELECT a.id, a.title, f.name AS source, a.published_at
    FROM articles a
    JOIN feeds f ON a.feed_id = f.id
    ORDER BY a.published_at DESC
""")

print("📰 Articles already in the database (from init.sql):")
print("=" * 80)
for row in cursor.fetchall():
    pub = row['published_at'].strftime('%Y-%m-%d %H:%M') if row['published_at'] else 'N/A'
    print(f"  [{row['id']:>2}] {row['title'][:55]:<55} | {row['source']:<18} | {pub}")

conn.close()

📰 Articles already in the database (from init.sql):
  [ 2] GPT-5 Launches With Major Reasoning Improvements        | Ars Technica       | 2026-04-19 23:24
  [127] Follow live: Pistons host Magic in Game 1               | ESPN Top Headlines | 2026-04-19 22:50
  [124] Be like Mike: With Jordan watching, Reddick wins        | ESPN Top Headlines | 2026-04-19 22:40
  [118] Bengals extend Lawrence after blockbuster trade         | ESPN Top Headlines | 2026-04-19 22:40
  [119] NFL draft prospect Branch arrested in Georgia           | ESPN Top Headlines | 2026-04-19 22:40
  [120] Indiana DC claps back at Simpson, Alabama fans          | ESPN Top Headlines | 2026-04-19 22:40
  [121] 'Gonna get loud': Mets head home as skid hits 11        | ESPN Top Headlines | 2026-04-19 22:40
  [123] Fitzpatrick beats Scheffler in playoff at Heritage      | ESPN Top Headlines | 2026-04-19 22:40
  [125] Guardians' Hedges gets engaged on field after win       | ESPN Top Headlines | 2026-04-19 22:40
  [122] Sixer

In [5]:
def store_articles(articles: list[dict]) -> int:
    """
    Insert normalised articles into PostgreSQL.
    Returns the number of NEW articles inserted (skips duplicates by URL).
    """
    conn = get_db()
    cursor = conn.cursor()
    inserted = 0

    for article in articles:
        try:
            cursor.execute("""
                INSERT INTO articles
                    (feed_id, title, url, summary, author, published_at, content_hash, word_count)
                VALUES
                    (%(feed_id)s, %(title)s, %(url)s, %(summary)s, %(author)s,
                     %(published_at)s, %(content_hash)s, %(word_count)s)
                ON CONFLICT (url) DO NOTHING
            """, article)
            # rowcount = 1 if inserted, 0 if it was a duplicate
            inserted += cursor.rowcount
        except Exception as e:
            print(f"  ⚠️  Error inserting '{article['title'][:40]}': {e}")
            conn.rollback()
            continue

    conn.commit()
    conn.close()
    return inserted

# Crawl a live feed and store the articles
print("🕷️ Crawling Hacker News front page...")
parsed = feedparser.parse("https://hnrss.org/frontpage")

articles = [normalise_entry(e, feed_id=8) for e in parsed.entries]
new_count = store_articles(articles)

print(f"\n✅ Crawled {len(articles)} articles, inserted {new_count} new ones")
print(f"   (skipped {len(articles) - new_count} duplicates)")

🕷️ Crawling Hacker News front page...



✅ Crawled 20 articles, inserted 1 new ones
   (skipped 19 duplicates)


## ⏰ Scheduling Crawls with Redis

### 🚫 Bad approach: crawl everything, every minute

The naive version looks like:

```python
while True:
    for feed in ALL_FEEDS:
        crawl(feed)   # hits EVERY feed on EVERY loop
    time.sleep(60)
```

Why this is bad:
- Hammers the slow, low-priority sources just as often as breaking-news sources
- Most crawl attempts return **nothing new** — wasted bandwidth on both sides
- A broken feed keeps getting retried immediately, risking an IP ban
- No way to prioritise: a 30-minute-interval science blog blocks the 5-minute breaking-news feed

### ✅ Better: a priority queue of *next-crawl times*

Each feed has a different crawl frequency:
- Breaking news sites: every 5–10 minutes
- Blogs: every 30–60 minutes
- Low-priority sources: every few hours

We use a **Redis sorted set** as a priority queue. The score is the Unix timestamp when the feed should next be crawled. To find feeds that are "due", we ask Redis for all entries with a score ≤ now — a single O(log N) range query.

```
Redis Sorted Set: "crawl_schedule"
┌────────────────────────────────────────────┐
│  Member (feed_id)  │  Score (next_crawl)   │
├────────────────────┼───────────────────────┤
│  feed:4            │  1711800000  (now)     │
│  feed:5            │  1711800000  (now)     │
│  feed:1            │  1711800600  (+10 min) │
│  feed:3            │  1711802400  (+40 min) │
└────────────────────┴───────────────────────┘
```

In [6]:
r = get_redis()

def init_crawl_schedule():
    """
    Load all active feeds from PostgreSQL and schedule them in Redis.
    Each feed's next crawl time = now (crawl immediately on startup).
    """
    conn = get_db()
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cursor.execute("SELECT id, name, url, crawl_interval_minutes FROM feeds WHERE is_active = TRUE")
    feeds = cursor.fetchall()
    conn.close()

    now = time.time()
    for feed in feeds:
        # Score = when to crawl next (now = crawl immediately)
        r.zadd("crawl_schedule", {f"feed:{feed['id']}": now})
        # Store feed metadata for quick access
        r.hset(f"feed_meta:{feed['id']}", mapping={
            "name": feed["name"],
            "url": feed["url"],
            "interval": feed["crawl_interval_minutes"]
        })

    print(f"📅 Scheduled {len(feeds)} feeds for crawling")
    return feeds

feeds = init_crawl_schedule()

# Show the schedule
print("\n📋 Current crawl schedule:")
print("=" * 60)
schedule = r.zrangebyscore("crawl_schedule", "-inf", "+inf", withscores=True)
for member, score in schedule:
    feed_id = member.split(":")[1]
    meta = r.hgetall(f"feed_meta:{feed_id}")
    next_crawl = datetime.fromtimestamp(score).strftime("%H:%M:%S")
    print(f"  {meta.get('name', '?'):<25} | interval: {meta.get('interval', '?'):>3} min | next: {next_crawl}")

📅 Scheduled 8 feeds for crawling

📋 Current crawl schedule:
  TechCrunch                | interval:  15 min | next: 03:41:00
  Ars Technica              | interval:  15 min | next: 03:41:00
  NASA Breaking News        | interval:  30 min | next: 03:41:00
  BBC News World            | interval:  10 min | next: 03:41:00
  ESPN Top Headlines        | interval:  10 min | next: 03:41:00
  Reuters Business          | interval:  10 min | next: 03:41:00
  NPR Health                | interval:  30 min | next: 03:41:00
  Hacker News               | interval:  10 min | next: 03:41:00


In [7]:
def get_feeds_due_for_crawl() -> list:
    """
    Return all feeds whose next crawl time is <= now.
    Uses ZRANGEBYSCORE to efficiently find overdue feeds.
    """
    now = time.time()
    due = r.zrangebyscore("crawl_schedule", "-inf", now)
    return due

def mark_feed_crawled(feed_id: int, interval_minutes: int):
    """
    After crawling a feed, reschedule it for (now + interval).
    """
    next_crawl = time.time() + (interval_minutes * 60)
    r.zadd("crawl_schedule", {f"feed:{feed_id}": next_crawl})

# Check which feeds are due right now
due_feeds = get_feeds_due_for_crawl()
print(f"⏰ Feeds due for crawling: {len(due_feeds)}")
for member in due_feeds:
    feed_id = member.split(":")[1]
    meta = r.hgetall(f"feed_meta:{feed_id}")
    print(f"   → {meta.get('name', '?')} (every {meta.get('interval', '?')} min)")

⏰ Feeds due for crawling: 8
   → TechCrunch (every 15 min)
   → Ars Technica (every 15 min)
   → NASA Breaking News (every 30 min)
   → BBC News World (every 10 min)
   → ESPN Top Headlines (every 10 min)
   → Reuters Business (every 10 min)
   → NPR Health (every 30 min)
   → Hacker News (every 10 min)


## 🕷️ The Complete Crawler

Let's put it all together: a crawler that checks Redis for due feeds, fetches and parses them, stores articles in PostgreSQL, and reschedules the feed.

In production, this would run as a long-lived worker process. Here, we'll run **one crawl cycle** to see it in action.

In [8]:
def crawl_one_feed(feed_id: int, feed_url: str, feed_name: str) -> dict:
    """
    Crawl a single RSS feed:
    1. Fetch and parse the XML
    2. Normalise each article
    3. Store in PostgreSQL (skip duplicate URLs)
    Returns a summary of what happened.
    """
    start = time.time()
    result = {"feed": feed_name, "fetched": 0, "inserted": 0, "errors": []}

    # Step 1: Fetch
    parsed = feedparser.parse(feed_url)
    if parsed.bozo and not parsed.entries:
        result["errors"].append(str(parsed.bozo_exception))
        return result

    # Step 2: Normalise
    articles = []
    for entry in parsed.entries:
        try:
            articles.append(normalise_entry(entry, feed_id))
        except Exception as e:
            result["errors"].append(f"Parse error: {e}")

    result["fetched"] = len(articles)

    # Step 3: Store
    if articles:
        result["inserted"] = store_articles(articles)

    result["duration_ms"] = round((time.time() - start) * 1000)
    return result


def run_crawl_cycle():
    """
    One complete crawl cycle:
    1. Find all feeds due for crawling
    2. Crawl each one
    3. Reschedule for next crawl
    """
    due = get_feeds_due_for_crawl()
    print(f"🕷️ Starting crawl cycle — {len(due)} feeds due")
    print("=" * 70)

    total_fetched = 0
    total_inserted = 0

    for member in due:
        feed_id = int(member.split(":")[1])
        meta = r.hgetall(f"feed_meta:{feed_id}")
        feed_url = meta.get("url", "")
        feed_name = meta.get("name", "Unknown")
        interval = int(meta.get("interval", 15))

        result = crawl_one_feed(feed_id, feed_url, feed_name)

        status = "✅" if not result["errors"] else "⚠️"
        print(f"  {status} {result['feed']:<25} | "
              f"fetched: {result['fetched']:>3} | "
              f"new: {result['inserted']:>3} | "
              f"{result.get('duration_ms', 0)} ms")
        if result["errors"]:
            for err in result["errors"][:2]:
                print(f"       ↳ {err[:80]}")

        total_fetched += result["fetched"]
        total_inserted += result["inserted"]

        # Reschedule this feed
        mark_feed_crawled(feed_id, interval)

    print()
    print(f"📊 Cycle complete: {total_fetched} articles fetched, {total_inserted} new")

# Run one crawl cycle
run_crawl_cycle()

🕷️ Starting crawl cycle — 8 feeds due


  ✅ TechCrunch                | fetched:  20 | new:   0 | 260 ms


  ✅ Ars Technica              | fetched:  20 | new:   0 | 1039 ms


  ✅ NASA Breaking News        | fetched:  10 | new:   0 | 378 ms
  ✅ BBC News World            | fetched:  33 | new:   0 | 201 ms


  ✅ ESPN Top Headlines        | fetched:  29 | new:   1 | 601 ms


  ⚠️ Reuters Business          | fetched:   0 | new:   0 | 0 ms
       ↳ <unknown>:38:160: mismatched tag


  ⚠️ NPR Health                | fetched:   0 | new:   0 | 0 ms
       ↳ <unknown>:2:0: syntax error


  ✅ Hacker News               | fetched:  20 | new:   0 | 1767 ms

📊 Cycle complete: 132 articles fetched, 1 new


In [9]:
# Check the schedule after crawling — feeds should be pushed into the future
print("📋 Updated crawl schedule (after one cycle):")
print("=" * 60)
schedule = r.zrangebyscore("crawl_schedule", "-inf", "+inf", withscores=True)
now = time.time()
for member, score in schedule:
    feed_id = member.split(":")[1]
    meta = r.hgetall(f"feed_meta:{feed_id}")
    minutes_until = (score - now) / 60
    print(f"  {meta.get('name', '?'):<25} | next crawl in {minutes_until:>5.1f} min")

# How many feeds are due right now? Should be 0.
due_now = get_feeds_due_for_crawl()
print(f"\n⏰ Feeds due right now: {len(due_now)} (should be 0 — all were just crawled)")

📋 Updated crawl schedule (after one cycle):
  BBC News World            | next crawl in   9.9 min
  ESPN Top Headlines        | next crawl in  10.0 min
  Reuters Business          | next crawl in  10.0 min
  Hacker News               | next crawl in  10.0 min
  TechCrunch                | next crawl in  14.9 min
  Ars Technica              | next crawl in  14.9 min
  NASA Breaking News        | next crawl in  29.9 min
  NPR Health                | next crawl in  30.0 min

⏰ Feeds due right now: 0 (should be 0 — all were just crawled)


## 🛡️ Handling Failures

In production, feeds break all the time:
- The server is down (HTTP 500)
- The feed URL changed (HTTP 404)
- The server rate-limits us (HTTP 429)
- The XML is malformed

A good crawler uses **exponential back-off**: if a feed fails, wait longer before retrying (double the wait each time). This prevents hammering a broken server.

```
Attempt 1: failed → wait 1 minute
Attempt 2: failed → wait 2 minutes
Attempt 3: failed → wait 4 minutes
Attempt 4: failed → wait 8 minutes
...
Max: wait 60 minutes, then try again
```

In [10]:
def schedule_with_backoff(feed_id: int, base_interval: int, failure_count: int):
    """
    After a failed crawl, schedule the next attempt with exponential back-off.
    
    base_interval: normal crawl interval in minutes
    failure_count: how many consecutive failures
    """
    # Exponential back-off: base * 2^failures, capped at 60 minutes
    backoff_minutes = min(base_interval * (2 ** failure_count), 60)
    next_crawl = time.time() + (backoff_minutes * 60)

    r.zadd("crawl_schedule", {f"feed:{feed_id}": next_crawl})
    # Track failure count in Redis
    r.hset(f"feed_meta:{feed_id}", "failures", failure_count)

    return backoff_minutes

# Demonstrate back-off progression
print("📉 Exponential back-off for a feed with 10-minute interval:")
print("=" * 50)
for failures in range(6):
    wait = min(10 * (2 ** failures), 60)
    bar = "█" * (wait // 2)
    print(f"  Failure #{failures}: wait {wait:>3} min  {bar}")

print("\n💡 After the feed recovers, reset failure count to 0")
print("   and go back to the normal crawl interval.")

📉 Exponential back-off for a feed with 10-minute interval:
  Failure #0: wait  10 min  █████
  Failure #1: wait  20 min  ██████████
  Failure #2: wait  40 min  ████████████████████
  Failure #3: wait  60 min  ██████████████████████████████
  Failure #4: wait  60 min  ██████████████████████████████
  Failure #5: wait  60 min  ██████████████████████████████

💡 After the feed recovers, reset failure count to 0
   and go back to the normal crawl interval.


## 🤝 Being a Polite Crawler

A real crawler lives on the open internet and must behave like a *good citizen*. Three cheap wins cut bandwidth (and the risk of being rate-limited) dramatically:

### 1. Conditional GET with `If-Modified-Since` / `ETag`

On every fetch, remember the server's `Last-Modified` header or `ETag`. On the next fetch, send them back — if nothing changed, the server returns `304 Not Modified` with **no body**, saving bandwidth on both ends.

```
GET /feed/rss.xml HTTP/1.1
If-Modified-Since: Sun, 20 Apr 2026 03:00:00 GMT
If-None-Match: "abc123"

──►  304 Not Modified   (tiny response, no parsing needed)
```

### 2. Respect `robots.txt`
Most sites publish `robots.txt` listing paths you should not crawl and recommended `Crawl-delay` values. Python's `urllib.robotparser` reads it for you.

### 3. Identify yourself
Set a descriptive `User-Agent` such as `MyNewsAggregator/1.0 (+https://example.com/bot)`. Site owners can then contact you instead of silently blocking.

### Demo: what 304 saves you

In [11]:
# feedparser supports conditional GET out of the box via `etag` / `modified` args.
# We simulate two back-to-back fetches to show the 304 response.

feed_url = "https://hnrss.org/frontpage"

# First fetch: record the server's freshness tokens
first = feedparser.parse(feed_url)
etag = first.get("etag")
modified = first.get("modified")
print(f"First fetch: {len(first.entries)} entries, status {first.get('status')}")
print(f"  etag={etag!r}")
print(f"  modified={modified!r}")

# Second fetch: pass them back. Servers that support conditional GET reply 304.
second = feedparser.parse(feed_url, etag=etag, modified=modified)
print(f"\nSecond fetch: status {second.get('status')}, {len(second.entries)} entries")
if second.get('status') == 304:
    print("  ✅ 304 Not Modified — server sent *no body*, saving bandwidth.")
else:
    print("  ℹ️  This server didn't return 304, or the feed changed between fetches.")
    print("      Most large publishers (BBC, NYT, …) honour conditional GET.")

First fetch: 20 entries, status 200
  etag=None
  modified='Sun, 19 Apr 2026 22:56:32 GMT'



Second fetch: status 304, 0 entries
  ✅ 304 Not Modified — server sent *no body*, saving bandwidth.


## 🧹 Cleanup

In [12]:
r = get_redis()
# Clean up Redis keys from this notebook
keys_to_delete = r.keys("crawl_schedule") + r.keys("feed_meta:*")
if keys_to_delete:
    r.delete(*keys_to_delete)
    print(f"🧹 Cleaned up {len(keys_to_delete)} Redis keys")
else:
    print("🧹 Nothing to clean up")

🧹 Cleaned up 9 Redis keys


## 📚 Summary

### Key Takeaways

1. **RSS feeds** give us structured article data — no scraping needed
2. **Normalisation** is critical — different feeds use different formats for dates, authors, and summaries
3. **`ON CONFLICT DO NOTHING`** prevents duplicate rows when re-crawling
4. **Redis sorted sets** make excellent priority queues for crawl scheduling
5. **Exponential back-off** protects both the aggregator and the source from failures
6. **Conditional GET** (`If-Modified-Since` / `ETag`) turns most re-fetches into cheap 304s

### System Design Interview Tips

- Mention that real aggregators crawl **thousands of feeds** — this is a perfect use case for a distributed task queue
- Discuss **crawl politeness**: respect `robots.txt`, use `If-Modified-Since` headers, don't hammer servers
- Point out that high-priority feeds (breaking news) get shorter intervals than blogs

### Next Up

In **Notebook 2**, we'll tackle **Content Deduplication & Ranking** — detecting that five outlets reported the same story and deciding which articles should appear at the top.